<a href="https://colab.research.google.com/github/JiHoonPark96/practice/blob/main/Multi_1and2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# =========================================================
# Homework 1 - PCA on an Image
# Clean PDF version for Google Colab
# Student Name: Ji Hoon Park
# =========================================================

# ---------------------------
# 0. Install required package
# ---------------------------
!pip -q install reportlab

# ---------------------------
# 1. Imports
# ---------------------------
import os
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from PIL import Image

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, Image as RLImage, Preformatted
)

# ---------------------------
# 2. Student info
# ---------------------------
student_name = "Ji Hoon Park"
output_pdf = f"{student_name.replace(' ', '_')}_Homework1_PCA_Image.pdf"
appendix_code_txt = "appendix_code_homework1.txt"

# ---------------------------
# 3. Upload image
# ---------------------------
print("Please upload one image file (jpg, jpeg, png, bmp).")
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded.")

image_filename = list(uploaded.keys())[0]
print(f"Uploaded file: {image_filename}")

# ---------------------------
# 4. Load image and convert to grayscale
# ---------------------------
img = Image.open(image_filename).convert("L")
img_array = np.array(img, dtype=np.float64)
img_norm = img_array / 255.0
m, n = img_norm.shape

# ---------------------------
# 5. SVD (used for PCA-style image reconstruction)
# ---------------------------
U, s, Vt = np.linalg.svd(img_norm, full_matrices=False)
rank_max = len(s)

# ---------------------------
# 6. Helper functions
# ---------------------------
def reconstruct_top_k(U, s, Vt, k):
    k = max(1, min(k, len(s)))
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

def reconstruct_range(U, s, Vt, start_idx, end_idx):
    start_idx = max(1, start_idx)
    end_idx = min(len(s), end_idx)
    if start_idx > end_idx:
        raise ValueError("Invalid range for singular values.")
    start0 = start_idx - 1
    end0 = end_idx
    return U[:, start0:end0] @ np.diag(s[start0:end0]) @ Vt[start0:end0, :]

def clip01(x):
    return np.clip(x, 0, 1)

def mse(a, b):
    return np.mean((a - b) ** 2)

def explained_variance_ratio_from_svd(s):
    eigvals = s**2
    return eigvals / np.sum(eigvals)

# ---------------------------
# 7. Requested reconstructions
# ---------------------------
k2 = min(2, rank_max)
k10 = min(10, rank_max)
k100 = min(100, rank_max)
khalf = max(1, rank_max // 2)
kall = rank_max

img_top2 = clip01(reconstruct_top_k(U, s, Vt, k2))
img_top10 = clip01(reconstruct_top_k(U, s, Vt, k10))
img_11_100 = clip01(reconstruct_range(U, s, Vt, 11, 100)) if rank_max >= 11 else np.zeros_like(img_norm)
img_top100 = clip01(reconstruct_top_k(U, s, Vt, k100))
img_half = clip01(reconstruct_top_k(U, s, Vt, khalf))
img_all = clip01(reconstruct_top_k(U, s, Vt, kall))

# ---------------------------
# 8. Explained variance + summary table
# ---------------------------
evr = explained_variance_ratio_from_svd(s)
cum_evr = np.cumsum(evr)

summary_df = {
    "Version": [
        "Top 2 Eigenvalues",
        "Top 10 Eigenvalues",
        "11th through 100th",
        "Top 100 Eigenvalues",
        "First Half of Eigenvalues",
        "All Eigenvalues"
    ],
    "Components Used": [
        f"1-{k2}",
        f"1-{k10}",
        "11-100" if rank_max >= 11 else "Not available",
        f"1-{k100}",
        f"1-{khalf}",
        f"1-{kall}"
    ],
    "MSE": [
        round(mse(img_norm, img_top2), 6),
        round(mse(img_norm, img_top10), 6),
        round(mse(img_norm, img_11_100), 6) if rank_max >= 11 else "N/A",
        round(mse(img_norm, img_top100), 6),
        round(mse(img_norm, img_half), 6),
        round(mse(img_norm, img_all), 6),
    ]
}

variance_info = {
    "Top 2 Eigenvalues": f"{cum_evr[k2-1]*100:.2f}%",
    "Top 10 Eigenvalues": f"{cum_evr[k10-1]*100:.2f}%",
    "Top 100 Eigenvalues": f"{cum_evr[k100-1]*100:.2f}%",
    "First Half of Eigenvalues": f"{cum_evr[khalf-1]*100:.2f}%",
    "All Eigenvalues": f"{cum_evr[kall-1]*100:.2f}%"
}

# ---------------------------
# 9. English interpretation
# ---------------------------
comparison_text = f"""
The reconstruction using the top 2 eigenvalues preserves only the broadest structure of the image
and loses most fine details. The top 10 eigenvalues provide a much clearer approximation,
although some textures and edges are still blurred.

The reconstruction using the 11th through 100th eigenvalues mainly captures secondary details
rather than the main overall structure, so it does not look like a complete image by itself.
The top 100 eigenvalues retain substantially more visual information and produce a much better approximation.

Using the first half of the eigenvalues results in a reconstruction that is visually very close
to the original image. Finally, using all eigenvalues reproduces the original image almost exactly, as expected.
""".strip()

method_text = """
This report applies principal component analysis to an image through Singular Value Decomposition (SVD).
The uploaded image was converted to grayscale and represented as a matrix.
Different rank-k approximations were then constructed using selected singular values.

For image data, the largest singular values capture the most important variation in the image.
Thus, keeping only a subset of the largest singular values provides a compressed approximation of the original image.
""".strip()

conclusion_text = f"""
The image can be summarized effectively with a relatively small number of components.
In this case, the first 10 eigenvalues already preserve much of the visible structure,
while the first 100 eigenvalues produce a high-quality reconstruction.
Using half of the available eigenvalues makes the reconstructed image very close to the original,
and using all eigenvalues reproduces the image almost exactly.
""".strip()

# ---------------------------
# 10. Save figures
# ---------------------------

# (a) Comparison image grid
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.ravel()

images = [
    (img_norm, "Original"),
    (img_top2, "Top 2 Eigenvalues"),
    (img_top10, "Top 10 Eigenvalues"),
    (img_11_100, "11th through 100th"),
    (img_top100, "Top 100 Eigenvalues"),
    (img_half, "First Half of Eigenvalues"),
    (img_all, "All Eigenvalues"),
]

for ax in axes:
    ax.axis("off")

for i, (im, title) in enumerate(images):
    axes[i].imshow(im, cmap="gray")
    axes[i].set_title(title, fontsize=11)
    axes[i].axis("off")

axes[7].text(
    0.0, 1.0,
    f"Student Name: {student_name}\n\n"
    f"Simple Comparison\n\n"
    f"{comparison_text}",
    fontsize=10,
    va="top",
    wrap=True
)
axes[7].axis("off")

plt.tight_layout()
comparison_png = "homework1_comparison.png"
plt.savefig(comparison_png, dpi=300, bbox_inches="tight")
plt.close()

# (b) Scree plot
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(s)+1), s, marker='o', linewidth=1.2, markersize=3)
plt.xlabel("Component Index")
plt.ylabel("Singular Value")
plt.title("Scree Plot")
plt.grid(alpha=0.3)
plt.tight_layout()
scree_png = "homework1_scree_plot.png"
plt.savefig(scree_png, dpi=300)
plt.close()

# (c) Cumulative explained variance
plt.figure(figsize=(8, 5))
plt.plot(np.arange(1, len(cum_evr)+1), cum_evr, marker='o', linewidth=1.2, markersize=3)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance Ratio")
plt.title("Cumulative Explained Variance")
plt.ylim(0, 1.02)
plt.grid(alpha=0.3)
plt.tight_layout()
cumvar_png = "homework1_cumulative_variance.png"
plt.savefig(cumvar_png, dpi=300)
plt.close()

# ---------------------------
# 11. Save appendix code text
# ---------------------------
code_text = r'''
# Main Analysis Steps
# 1. Upload one image
# 2. Convert the image to grayscale
# 3. Compute SVD of the image matrix
# 4. Reconstruct the image using:
#    - top 2 eigenvalues
#    - top 10 eigenvalues
#    - 11th through 100th eigenvalues
#    - top 100 eigenvalues
#    - first half of the eigenvalues
#    - all eigenvalues
# 5. Compute MSE values
# 6. Plot comparison figure, scree plot, and cumulative explained variance
# 7. Export all results to a single PDF report
'''
with open(appendix_code_txt, "w", encoding="utf-8") as f:
    f.write(code_text)

# ---------------------------
# 12. ReportLab styles
# ---------------------------
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "TitleStyle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=18,
    leading=22,
    alignment=TA_CENTER,
    spaceAfter=12
)

subtitle_style = ParagraphStyle(
    "SubtitleStyle",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=12,
    leading=15,
    textColor=colors.HexColor("#1F3A5F"),
    spaceBefore=8,
    spaceAfter=6
)

body_style = ParagraphStyle(
    "BodyStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=10.5,
    leading=14,
    alignment=TA_LEFT,
    spaceAfter=6
)

mono_style = ParagraphStyle(
    "MonoStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=8.5,
    leading=10
)

# ---------------------------
# 13. Helper for pretty tables
# ---------------------------
def make_reportlab_table(data_rows, col_widths=None, font_size=9):
    table = Table(data_rows, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAF7")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), font_size),
        ("LEADING", (0, 0), (-1, -1), font_size + 2),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7FBFE")]),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ]))
    return table

# ---------------------------
# 14. Build summary table for PDF
# ---------------------------
summary_table_rows = [
    ["Version", "Components Used", "MSE"]
]

for i in range(len(summary_df["Version"])):
    summary_table_rows.append([
        summary_df["Version"][i],
        str(summary_df["Components Used"][i]),
        str(summary_df["MSE"][i])
    ])

variance_table_rows = [
    ["Category", "Cumulative Explained Variance"],
    ["Top 2 Eigenvalues", variance_info["Top 2 Eigenvalues"]],
    ["Top 10 Eigenvalues", variance_info["Top 10 Eigenvalues"]],
    ["Top 100 Eigenvalues", variance_info["Top 100 Eigenvalues"]],
    ["First Half of Eigenvalues", variance_info["First Half of Eigenvalues"]],
    ["All Eigenvalues", variance_info["All Eigenvalues"]],
]

# ---------------------------
# 15. Build PDF
# ---------------------------
doc = SimpleDocTemplate(
    output_pdf,
    pagesize=A4,
    rightMargin=1.7*cm,
    leftMargin=1.7*cm,
    topMargin=1.6*cm,
    bottomMargin=1.6*cm
)

story = []

# Page 1: Cover / summary
story.append(Paragraph("Homework 1: Principal Component Analysis of an Image", title_style))
story.append(Paragraph(f"<b>Student Name:</b> {student_name}", body_style))
story.append(Spacer(1, 0.2*cm))

story.append(Paragraph("Data and Method", subtitle_style))
story.append(Paragraph(
    f"The uploaded image was converted to grayscale and represented as a matrix of size {m} × {n}. "
    f"Singular Value Decomposition (SVD) was used to compute the principal components of the image.",
    body_style
))
story.append(Paragraph(method_text, body_style))

story.append(Paragraph("Key Result", subtitle_style))
story.append(Paragraph(conclusion_text, body_style))

story.append(Paragraph("Simple Comparison", subtitle_style))
story.append(Paragraph(comparison_text.replace("\n", "<br/>"), body_style))

story.append(PageBreak())

# Page 2: summary tables
story.append(Paragraph("Reconstruction Summary", subtitle_style))
story.append(make_reportlab_table(summary_table_rows, col_widths=[7*cm, 4.2*cm, 4*cm], font_size=9))
story.append(Spacer(1, 0.5*cm))

story.append(Paragraph("Cumulative Explained Variance", subtitle_style))
story.append(make_reportlab_table(variance_table_rows, col_widths=[8*cm, 7*cm], font_size=9))

story.append(Spacer(1, 0.5*cm))
story.append(Paragraph("Image Information", subtitle_style))
story.append(Paragraph(
    f"Original image size: {m} × {n}<br/>"
    f"Maximum available number of eigenvalues: {rank_max}",
    body_style
))

story.append(PageBreak())

# Page 3: comparison image
story.append(Paragraph("Figure 1. PCA Reconstruction Comparison", subtitle_style))
story.append(RLImage(comparison_png, width=17*cm, height=8.8*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "This figure compares the original image with several PCA-based reconstructions using different numbers of eigenvalues.",
    body_style
))

story.append(PageBreak())

# Page 4: scree plot
story.append(Paragraph("Figure 2. Scree Plot", subtitle_style))
story.append(RLImage(scree_png, width=16.5*cm, height=10.3*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "The scree plot shows the magnitude of the singular values across components. "
    "The largest components contain most of the important image information.",
    body_style
))

story.append(PageBreak())

# Page 5: cumulative explained variance
story.append(Paragraph("Figure 3. Cumulative Explained Variance", subtitle_style))
story.append(RLImage(cumvar_png, width=16.5*cm, height=10.3*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "This figure shows how the cumulative explained variance increases as more components are included.",
    body_style
))

story.append(PageBreak())

# Appendix
story.append(Paragraph("Appendix", subtitle_style))
story.append(Paragraph("Appendix A. Code Summary", body_style))

with open(appendix_code_txt, "r", encoding="utf-8") as f:
    appendix_code_content = f.read()

story.append(Preformatted(appendix_code_content, mono_style))
story.append(Spacer(1, 0.5*cm))

story.append(Paragraph("Appendix B. Notes", body_style))
story.append(Paragraph(
    "For image PCA, singular values from SVD correspond to the principal directions of variation. "
    "In practical image compression, keeping the largest singular values is equivalent to retaining the most important principal components.",
    body_style
))

doc.build(story)

print(f"PDF created successfully: {output_pdf}")

# ---------------------------
# 16. Download PDF
# ---------------------------
files.download(output_pdf)

Please upload one image file (jpg, jpeg, png, bmp).


Saving 20161115182040477_QAMZS182.png to 20161115182040477_QAMZS182 (1).png
Uploaded file: 20161115182040477_QAMZS182 (1).png
PDF created successfully: Ji_Hoon_Park_Homework1_PCA_Image.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# =========================================================
# Homework 2 - Q4.5 PCA on IRIS.xlsx
# Clean PDF version for Google Colab
# Student Name: Ji Hoon Park
# =========================================================

# ---------------------------
# 0. Install required package
# ---------------------------
!pip -q install reportlab openpyxl

# ---------------------------
# 1. Imports
# ---------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from reportlab.lib import colors
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.lib.units import cm
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
    PageBreak, Image as RLImage, Preformatted
)

# ---------------------------
# 2. Student info
# ---------------------------
student_name = "Ji Hoon Park"
output_pdf = f"{student_name.replace(' ', '_')}_Homework2_Q45_PCA_IRIS.pdf"
appendix_code_txt = "appendix_code_homework2_q45.txt"

# ---------------------------
# 3. Upload file
# ---------------------------
print("Please upload IRIS.xlsx")
uploaded = files.upload()

if len(uploaded) == 0:
    raise ValueError("No file was uploaded.")

file_name = list(uploaded.keys())[0]
print(f"Uploaded file: {file_name}")

# ---------------------------
# 4. Read data
# ---------------------------
df = pd.read_excel(file_name)

required_cols = ["X1", "X2", "X3", "X4", "X5"]
for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Column {c} is missing from the uploaded file.")

species_map = {
    1: "Iris setosa",
    2: "Iris versicolor",
    3: "Iris virginica"
}
df["Species"] = df["X1"].map(species_map)

feature_cols = ["X2", "X3", "X4", "X5"]
feature_names = {
    "X2": "Sepal Length",
    "X3": "Sepal Width",
    "X4": "Petal Length",
    "X5": "Petal Width"
}

X = df[feature_cols].copy()

# ---------------------------
# 5. Standardize and run PCA
# ---------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA()
scores = pca.fit_transform(X_scaled)

explained_variance_ratio = pca.explained_variance_ratio_
eigenvalues = pca.explained_variance_
cum_explained = np.cumsum(explained_variance_ratio)

loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_cols,
    columns=[f"PC{i+1}" for i in range(len(feature_cols))]
)

df["PC1"] = scores[:, 0]
df["PC2"] = scores[:, 1]

variance_table = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(len(feature_cols))],
    "Eigenvalue": np.round(eigenvalues, 4),
    "Explained Variance Ratio": np.round(explained_variance_ratio, 4),
    "Cumulative Explained Variance": np.round(cum_explained, 4)
})

loadings_named = loadings.copy()
loadings_named.index = [feature_names[x] for x in loadings_named.index]
loadings_named = loadings_named.round(4)

species_scores = df.groupby("Species")[["PC1", "PC2"]].mean().reset_index().round(4)

# ---------------------------
# 6. Interpretation helper
# ---------------------------
def get_pc_interpretation(pc_col, top_n=4):
    s = loadings[pc_col].copy()
    ordered = s.abs().sort_values(ascending=False).index.tolist()
    parts = []
    for var in ordered[:top_n]:
        sign = "positive" if s[var] > 0 else "negative"
        parts.append(f"{feature_names[var]} ({sign}, loading={s[var]:.3f})")
    return parts

pc1_items = get_pc_interpretation("PC1")
pc2_items = get_pc_interpretation("PC2")

pcs_needed_80 = int(np.argmax(cum_explained >= 0.80) + 1)
pcs_needed_90 = int(np.argmax(cum_explained >= 0.90) + 1)
pcs_needed_95 = int(np.argmax(cum_explained >= 0.95) + 1)

pc1_var = explained_variance_ratio[0] * 100
pc2_var = explained_variance_ratio[1] * 100
pc12_var = (explained_variance_ratio[0] + explained_variance_ratio[1]) * 100

# ---------------------------
# 7. Save plots
# ---------------------------

# Scree plot
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(feature_cols)+1), eigenvalues, marker='o')
plt.xticks(range(1, len(feature_cols)+1))
plt.xlabel("Principal Component")
plt.ylabel("Eigenvalue")
plt.title("Scree Plot")
plt.grid(alpha=0.3)
plt.tight_layout()
scree_file = "scree_plot.png"
plt.savefig(scree_file, dpi=300)
plt.close()

# Cumulative explained variance
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(feature_cols)+1), cum_explained, marker='o')
plt.xticks(range(1, len(feature_cols)+1))
plt.ylim(0, 1.05)
plt.xlabel("Number of Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Explained Variance")
plt.grid(alpha=0.3)
plt.tight_layout()
cumvar_file = "cumulative_variance.png"
plt.savefig(cumvar_file, dpi=300)
plt.close()

# Score plot
plt.figure(figsize=(8, 6))
for sp in df["Species"].dropna().unique():
    subset = df[df["Species"] == sp]
    plt.scatter(subset["PC1"], subset["PC2"], label=sp, s=45)

for _, row in species_scores.iterrows():
    plt.text(row["PC1"], row["PC2"], row["Species"], fontsize=9)

plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.xlabel("PC1 Score")
plt.ylabel("PC2 Score")
plt.title("Scores on the First Two Principal Components")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
scoreplot_file = "pc_scores_scatter.png"
plt.savefig(scoreplot_file, dpi=300)
plt.close()

# Loading plot
plt.figure(figsize=(8, 6))
for var in feature_cols:
    plt.arrow(
        0, 0,
        loadings.loc[var, "PC1"],
        loadings.loc[var, "PC2"],
        head_width=0.03,
        length_includes_head=True
    )
    plt.text(
        loadings.loc[var, "PC1"] * 1.08,
        loadings.loc[var, "PC2"] * 1.08,
        feature_names[var],
        fontsize=10
    )

plt.axhline(0, color='gray', linewidth=0.8)
plt.axvline(0, color='gray', linewidth=0.8)
plt.xlabel("PC1 Loading")
plt.ylabel("PC2 Loading")
plt.title("Loading Plot for PC1 and PC2")
plt.grid(alpha=0.3)
plt.tight_layout()
loadingplot_file = "loading_plot.png"
plt.savefig(loadingplot_file, dpi=300)
plt.close()

# ---------------------------
# 8. Build clean English write-up
# ---------------------------
interpretation_text = f"""
The first principal component (PC1) explains {pc1_var:.2f}% of the total variance,
and the second principal component (PC2) explains {pc2_var:.2f}%.
Together, the first two principal components explain {pc12_var:.2f}% of the total variance.

Based on the cumulative explained variance, {pcs_needed_80} component(s) are needed
to explain at least 80% of the variance, {pcs_needed_90} component(s) are needed
to explain at least 90% of the variance, and {pcs_needed_95} component(s) are needed
to explain at least 95% of the variance.

PC1 is mainly associated with {", ".join(pc1_items)}.
PC2 is mainly associated with {", ".join(pc2_items)}.

When the observations are plotted in the PC1-PC2 plane, the three iris species show
distinct clustering patterns. This suggests that the first two principal components
capture important differences among species.

Overall, a small number of principal components is sufficient to summarize most of the
variation in the IRIS data, and the first two principal components provide a useful
low-dimensional representation of the data.
""".strip()

short_answer_a = f"""
(a) Two principal components are enough to adequately describe the data for most practical purposes,
because PC1 and PC2 together explain {pc12_var:.2f}% of the total variance.
""".strip()

short_answer_b = """
(b) The score plot of PC1 and PC2 shows clustering by species.
Iris setosa is typically well separated from the other two species, while
Iris versicolor and Iris virginica are closer but still show noticeable differences.
""".strip()

# ---------------------------
# 9. Save appendix code text
# ---------------------------
code_text = r'''
# Main Analysis Steps
# 1. Read IRIS.xlsx
# 2. Use X2-X5 for PCA
# 3. Standardize variables
# 4. Fit PCA
# 5. Extract explained variance, loadings, and scores
# 6. Plot scree plot, cumulative variance, score plot, and loading plot
# 7. Export all results to a single PDF report
'''
with open(appendix_code_txt, "w", encoding="utf-8") as f:
    f.write(code_text)

# ---------------------------
# 10. ReportLab styles
# ---------------------------
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    "TitleStyle",
    parent=styles["Title"],
    fontName="Helvetica-Bold",
    fontSize=18,
    leading=22,
    alignment=TA_CENTER,
    spaceAfter=12
)

subtitle_style = ParagraphStyle(
    "SubtitleStyle",
    parent=styles["Heading2"],
    fontName="Helvetica-Bold",
    fontSize=12,
    leading=15,
    textColor=colors.HexColor("#1F3A5F"),
    spaceBefore=8,
    spaceAfter=6
)

body_style = ParagraphStyle(
    "BodyStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=10.5,
    leading=14,
    alignment=TA_LEFT,
    spaceAfter=6
)

small_style = ParagraphStyle(
    "SmallStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=9,
    leading=11
)

mono_style = ParagraphStyle(
    "MonoStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=8.5,
    leading=10
)

# ---------------------------
# 11. Helper for pretty tables
# ---------------------------
def make_reportlab_table(df_input, col_widths=None, font_size=9):
    data = [list(df_input.columns)] + df_input.astype(str).values.tolist()
    table = Table(data, colWidths=col_widths, repeatRows=1)

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAF7")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), font_size),
        ("LEADING", (0, 0), (-1, -1), font_size + 2),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("ALIGN", (0, 0), (-1, -1), "CENTER"),
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7FBFE")]),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
        ("LEFTPADDING", (0, 0), (-1, -1), 5),
        ("RIGHTPADDING", (0, 0), (-1, -1), 5),
    ]))
    return table

# ---------------------------
# 12. Build PDF content
# ---------------------------
doc = SimpleDocTemplate(
    output_pdf,
    pagesize=A4,
    rightMargin=1.7*cm,
    leftMargin=1.7*cm,
    topMargin=1.6*cm,
    bottomMargin=1.6*cm
)

story = []

# Page 1: Cover / summary
story.append(Paragraph("Homework 2 - Q4.5 Principal Components Analysis of the IRIS Data", title_style))
story.append(Paragraph(f"<b>Student Name:</b> {student_name}", body_style))
story.append(Spacer(1, 0.2*cm))

story.append(Paragraph("Data Description", subtitle_style))
story.append(Paragraph(
    "The PCA was conducted using the four quantitative variables X2-X5: "
    "Sepal Length, Sepal Width, Petal Length, and Petal Width. "
    "The species indicator X1 was not included in the PCA itself, "
    "but it was used afterward to compare the three iris species in the space of the first two principal components.",
    body_style
))

story.append(Paragraph("Answer to Q4.5(a)", subtitle_style))
story.append(Paragraph(short_answer_a, body_style))

story.append(Paragraph("Answer to Q4.5(b)", subtitle_style))
story.append(Paragraph(short_answer_b, body_style))

story.append(Paragraph("Interpretation", subtitle_style))
for paragraph in interpretation_text.split("\n\n"):
    story.append(Paragraph(paragraph.strip(), body_style))

story.append(PageBreak())

# Page 2: Variance table + loadings table
story.append(Paragraph("Variance Explained", subtitle_style))
story.append(make_reportlab_table(variance_table, col_widths=[3*cm, 3*cm, 4.2*cm, 5.2*cm], font_size=9))
story.append(Spacer(1, 0.5*cm))

story.append(Paragraph("Loadings Table", subtitle_style))
loadings_for_pdf = loadings_named.reset_index().rename(columns={"index": "Variable"})
story.append(make_reportlab_table(loadings_for_pdf, col_widths=[5*cm, 3*cm, 3*cm, 3*cm, 3*cm], font_size=9))

story.append(Spacer(1, 0.5*cm))
story.append(Paragraph("Average PC Scores by Species", subtitle_style))
story.append(make_reportlab_table(species_scores, col_widths=[6*cm, 4*cm, 4*cm], font_size=9))

story.append(PageBreak())

# Page 3: Scree plot
story.append(Paragraph("Figure 1. Scree Plot", subtitle_style))
story.append(RLImage(scree_file, width=16.5*cm, height=10.3*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "The scree plot shows how much variance is associated with each principal component. "
    "A steep drop after the first few components suggests that most of the information is concentrated in the first two components.",
    body_style
))

story.append(PageBreak())

# Page 4: cumulative variance
story.append(Paragraph("Figure 2. Cumulative Explained Variance", subtitle_style))
story.append(RLImage(cumvar_file, width=16.5*cm, height=10.3*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "This figure shows how quickly the total explained variance accumulates as more components are added. "
    "The first two principal components already capture most of the total variation.",
    body_style
))

story.append(PageBreak())

# Page 5: score plot
story.append(Paragraph("Figure 3. Scores on the First Two Principal Components", subtitle_style))
story.append(RLImage(scoreplot_file, width=16.5*cm, height=12*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "The score plot shows how observations from the three species are positioned in the low-dimensional PC1-PC2 space. "
    "Clear clustering indicates that the first two components capture important species differences.",
    body_style
))

story.append(PageBreak())

# Page 6: loading plot
story.append(Paragraph("Figure 4. Loading Plot", subtitle_style))
story.append(RLImage(loadingplot_file, width=16.5*cm, height=12*cm))
story.append(Spacer(1, 0.4*cm))
story.append(Paragraph(
    "The loading plot helps interpret the principal components by showing the contribution of each original variable to PC1 and PC2.",
    body_style
))

story.append(PageBreak())

# Appendix page
story.append(Paragraph("Appendix", subtitle_style))
story.append(Paragraph("Appendix A. Code Summary", body_style))

with open(appendix_code_txt, "r", encoding="utf-8") as f:
    appendix_code_content = f.read()

story.append(Preformatted(appendix_code_content, mono_style))
story.append(Spacer(1, 0.4*cm))

story.append(Paragraph("Appendix B. Key Numerical Results", body_style))
story.append(Paragraph(
    f"PC1 explained variance: {pc1_var:.2f}%<br/>"
    f"PC2 explained variance: {pc2_var:.2f}%<br/>"
    f"PC1 + PC2 explained variance: {pc12_var:.2f}%<br/>"
    f"Components needed for 80%: {pcs_needed_80}<br/>"
    f"Components needed for 90%: {pcs_needed_90}<br/>"
    f"Components needed for 95%: {pcs_needed_95}",
    body_style
))

# ---------------------------
# 13. Build PDF
# ---------------------------
doc.build(story)

print(f"PDF created successfully: {output_pdf}")

# ---------------------------
# 14. Download PDF
# ---------------------------
files.download(output_pdf)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 28.7 MB/s eta 0:00:00
Please upload IRIS.xlsx


Saving IRIS.xlsx to IRIS (2).xlsx
Uploaded file: IRIS (2).xlsx
PDF created successfully: Ji_Hoon_Park_Homework2_Q45_PCA_IRIS.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>